In [ ]:
import pandas as pd
import numpy as np
import os
os.environ["PYTHONHASHSEED"] = "42"
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, precision_recall_curve, auc
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import random
import re
import pickle
from captum.attr import IntegratedGradients

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 42
set_seed(seed)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
test_df = pd.read_csv("data/Round3/70/changed_test_seqs_new.csv")
val_test_df = pd.read_csv("data/Round3/70/changed_val_seqs_test.csv")

In [ ]:
seq_cols = ["CDR1 aligned", "CDR2 aligned", "CDR3 aligned"]
amino_acids = list("ACDEFGHIKLMNPQRSTVWY-")
add_position = True

In [ ]:
def concat_cdrs(row):
    return "".join(str(row[col]) for col in seq_cols)
test_df["CDRs"] = test_df.apply(concat_cdrs, axis=1)
test_lengths = test_df["CDRs"].str.len().iloc[0]
val_test_df["CDRs"] = val_test_df.apply(concat_cdrs, axis=1)
val_test_lengths = val_test_df["CDRs"].str.len().iloc[0]

In [ ]:
aa2idx = {aa: i for i, aa in enumerate(amino_acids)}
I = np.eye(len(amino_acids),dtype=np.float32) 

In [ ]:
def one_hot_encode(seq, add_position=True):
    feats = []
    L = len(seq)
    for i, a in enumerate(seq):
        v = I[aa2idx.get(a, aa2idx["-"])] 
        if add_position:
            v = np.concatenate((v, [i / L]))
        feats.append(v)

    return np.array(feats, dtype=np.float32)  


In [ ]:
def build_tensors(df):
    seqs = [torch.tensor(one_hot_encode(seq), dtype=torch.float32)
            for seq in df["CDRs"]]
    X = pad_sequence(seqs, batch_first=True)  
    y = torch.tensor(df["Label"].values, dtype=torch.long)
    return X, y

x_test_tensor, y_test_tensor   = build_tensors(test_df)
x_val_test_tensor, y_val_test_tensor = build_tensors(val_test_df)

In [ ]:
class ProteinCNN(nn.Module):
    def __init__(
        self,
        input_dim=22,
        num_classes=2,
        num_blocks=3,
        base_ch=64,
        kernel_size=7,
        dropout=0.3,
        hidden_dim=128
    ):
        super().__init__()

        layers = []
        in_ch = input_dim
        padding = kernel_size // 2

        channel_sizes = [base_ch * (2 ** i) for i in range(num_blocks)]

        for out_ch in channel_sizes:
            layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding))
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.ReLU())
            in_ch = out_ch

        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(dropout)

        self.fc1 = nn.Linear(channel_sizes[-1], hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)   
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
model_path = r"best_cdrs_cnn_optuna_final_trainval_70.pth"
model = ProteinCNN(
    input_dim=22,
    num_classes=2,
    base_ch=96,        
    num_blocks=3,     
    kernel_size=15,   
    dropout=0.4862528132298237,     
    hidden_dim=64      
)

state = torch.load(model_path, map_location=device)
model.load_state_dict(state)
model = model.to(device)
model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [ ]:
def evaluate_model_on_split(model, X_tensor, y_tensor, df, split_name, batch_size=32):
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in tqdm(dataloader, desc=f"Evaluating {split_name}"):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = np.argmax(logits.cpu().numpy(), axis=1)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.cpu().numpy())

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    roc_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else np.nan
    c_report = classification_report(all_labels, all_preds)
    p, r, _ = precision_recall_curve(all_labels, all_probs)
    pr_auc = auc(r, p)
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n--- {split_name.upper()} results ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:        {pr_auc:.4f}")
    print("Confusion matrix:")
    print(cm)
    print(c_report)

    results_df = df.copy()
    results_df["y_true"] = all_labels
    results_df["y_score"] = all_probs
    results_df["pred_label"] = all_preds

    metrics = {
        "split": split_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "classification_report": c_report
    }

    return results_df, metrics, all_labels, all_preds, all_probs


In [ ]:
# test set
test_results_df, test_metrics, y_test_true, y_test_pred, y_test_score = evaluate_model_on_split(
    model= model,
    X_tensor=x_test_tensor,
    y_tensor=y_test_tensor,
    df=test_df,
    split_name="test",
    batch_size = 32
)

# val_test set
val_test_results_df, val_test_metrics, y_val_test_true, y_val_test_pred, y_val_test_score = evaluate_model_on_split(
    model= model,
    X_tensor=x_val_test_tensor,
    y_tensor=y_val_test_tensor,
    df=val_test_df,
    split_name="val_test",
    batch_size = 32
)


Evaluating test:   0%|          | 0/1144 [00:00<?, ?it/s]


Evaluating test:   0%|          | 1/1144 [00:00<02:20,  8.16it/s]


Evaluating test:  13%|█▎        | 144/1144 [00:00<00:01, 771.68it/s]


Evaluating test:  25%|██▌       | 289/1144 [00:00<00:00, 1065.64it/s]


Evaluating test:  38%|███▊      | 434/1144 [00:00<00:00, 1212.22it/s]

...


Evaluating test: 100%|██████████| 1144/1144 [00:00<00:00, 1250.72it/s]


--- TEST results ---
Accuracy:  0.6099
Precision: 0.6068
Recall:    0.5576
F1:        0.5811
ROC-AUC:   0.6515
PR-AUC:        0.6286
Confusion matrix:
[[12415  6416]
 [ 7857  9901]]
              precision    recall  f1-score   support

           0       0.61      0.66      0.63     18831
           1       0.61      0.56      0.58     17758

    accuracy                           0.61     36589
   macro avg       0.61      0.61      0.61     36589
weighted avg       0.61      0.61      0.61     36589




Evaluating val_test:   0%|          | 0/85 [00:00<?, ?it/s]


Evaluating val_test: 100%|██████████| 85/85 [00:00<00:00, 1481.85it/s]


--- VAL_TEST results ---
Accuracy:  0.5974
Precision: 0.5595
Recall:    0.4988
F1:        0.5274
ROC-AUC:   0.6379
PR-AUC:        0.5775
Confusion matrix:
[[1014  481]
 [ 614  611]]
              precision    recall  f1-score   support

           0       0.62      0.68      0.65      1495
           1       0.56      0.50      0.53      1225

    accuracy                           0.60      2720
   macro avg       0.59      0.59      0.59      2720
weighted avg       0.59      0.60      0.59      2720



In [ ]:
def evaluate_at_k(y_true, y_scores, clusters, k=100, split_name="Test"):
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)
    clusters = np.asarray(clusters)

    assert len(y_true) == len(y_scores), "y_true and y_scores must have same length"
    assert len(y_true) == len(clusters), "y_true and clusters must have same length"

    k = min(k, len(y_true))
    top_idx = np.argsort(y_scores)[::-1][:k]
    y_topk = y_true[top_idx]

    precision_k = y_topk.sum() / k
    total_positives = y_true.sum()

    k_1000 = min(1000, len(y_true))
    top_idx_1000 = np.argsort(y_scores)[::-1][:k_1000]
    precision_1000 = y_true[top_idx_1000].sum() / k_1000

    positive_rate = total_positives / len(y_true)
    ef_k = (precision_k / positive_rate) if positive_rate > 0 else np.nan

    diversity_k = len(np.unique(clusters[top_idx]))
    total_unique_clusters = len(np.unique(clusters))
    normalized_diversity_k = (
        diversity_k / total_unique_clusters if total_unique_clusters > 0 else np.nan
    )

    print(f" {split_name.upper()} @ {k}")
    print(f"Precision@{k}:            {precision_k:.3f}")
    print(f"Precision@1000:           {precision_1000:.3f}")
    print(f"EF@{k}:                   {ef_k:.3f}" if not np.isnan(ef_k) else f"EF@{k}: NaN")
    print(f"Diversity@{k}:            {diversity_k}")
    print(
        f"Normalized Diversity@{k}: {normalized_diversity_k:.3f}"
        if not np.isnan(normalized_diversity_k)
        else f"Normalized Diversity@{k}: NaN"
    )

    return {
        "top_idx": top_idx,
        f"precision@{k}": precision_k,
        "precision@1000": precision_1000,
        f"ef@{k}": ef_k,
        f"diversity@{k}": diversity_k,
        f"normalized_diversity@{k}": normalized_diversity_k
    }

In [ ]:
clusters_test = test_df["Cluster_number"].to_numpy()           
clusters_val_test = val_test_df["Cluster_number"].to_numpy()   

test_at_100 = evaluate_at_k(
    y_true = test_results_df["y_true"],  
    y_scores = test_results_df["y_score"],  
    clusters = clusters_test,
    k=100,
    split_name = "Test"
)

val_test_at_100 = evaluate_at_k(
    y_true = val_test_results_df["y_true"],  
    y_scores = val_test_results_df["y_score"],  
    clusters = clusters_val_test,
    k=100,
    split_name="Val_Test"
)


================ TEST @ 100 ================

Precision@100:            0.820
Precision@1000:           0.793
EF@100:                   1.690
Diversity@100:            95
Normalized Diversity@100: 0.005

================ VAL_TEST @ 100 ================

Precision@100:            0.730
Precision@1000:           0.567
EF@100:                   1.621
Diversity@100:            91
Normalized Diversity@100: 0.040


In [ ]:
test_df["true_label"] = test_results_df["y_true"]
test_df["pred_label"] = test_results_df["pred_label"]
test_df["pred_prob_1"] = test_results_df["y_score"]

val_test_df["true_label"] = val_test_results_df["y_true"]
val_test_df["pred_label"] = val_test_results_df["pred_label"]
val_test_df["pred_prob_1"] = val_test_results_df["y_score"]


In [ ]:
anarcii_df = pd.read_csv(r"/scratch/5269326/projects/myproj/data/Round3/Cluster exp/final_EG-ANARCII_imgt_wide_fixed_filtered.csv")
anarcii_position_cols = [
    c for c in anarcii_df.columns 
    if re.match(r'^\d+[A-Z]?$', str(c))
]
print("Number of positions:", len(anarcii_position_cols))

Number of positions: 152


In [ ]:

def forward_func(x):
    return model(x)

ig = IntegratedGradients(forward_func)

In [ ]:
def get_obs_attr_from_cdrs(seq, target):
    x_np = one_hot_encode(seq, add_position=True)  
    x = torch.tensor(x_np, dtype=torch.float32, device=device).unsqueeze(0)
    x.requires_grad_(True)
    baseline_seq = torch.zeros_like(x, dtype=torch.float32)

    attr = ig.attribute(x, target=target, baselines = baseline_seq, n_steps=50)
    attr = attr.detach().cpu().numpy()[0]

    x_aa = x_np[:, :len(aa2idx)]
    attr_aa = attr[:, :len(aa2idx)]

    obs_attr = x_aa * attr_aa
    return obs_attr

In [ ]:
seq_ids = test_df["Nanobody_id"].tolist()
print("Number of test sequences:", len(seq_ids))

Number of test sequences: 36589


In [ ]:
for _, row in test_df.sample(5, random_state=42).iterrows():
    x = torch.tensor(one_hot_encode(row["CDRs"]), dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        p = torch.softmax(model(x), dim=1)[0, 1].item()
    print(row["Nanobody_id"], "saved:", row["pred_prob_1"], "recomputed:", p)

Nb_fa3857a0a782 saved: 0.4969860017299652 recomputed: 0.4969824552536011
Nb_dedf7b610e9a saved: 0.3074321448802948 recomputed: 0.307433545589447
Nb_da2ee2736a18 saved: 0.4174772799015045 recomputed: 0.4174768626689911
Nb_5bd57d511312 saved: 0.7124049067497253 recomputed: 0.7124013304710388
Nb_afac9c3b5ef8 saved: 0.6225606799125671 recomputed: 0.6225649118423462


In [ ]:
tp_df = test_df[
    (test_df["true_label"] == 1) &
    (test_df["pred_label"] == 1)
].copy()

tn_df = test_df[
    (test_df["true_label"] == 0) &
    (test_df["pred_label"] == 0)
].copy()

cols = [
    "Nanobody_id",
    "Sequence",
    "CDRs",
    "Round",
    "true_label",
    "pred_label",
    "pred_prob_1"
]

tp_df = tp_df[cols]
tn_df = tn_df[cols]
tp_df.to_csv("/scratch/5269326/projects/myproj/outputs/Round3/tp_sequences_70_test.csv", index=False)
tn_df.to_csv("/scratch/5269326/projects/myproj/outputs/Round3/tn_sequences_70_test.csv", index=False)

In [ ]:
positive_explanations = []   
negative_explanations = []   

for _, row in test_df.iterrows():
    seq_id = row.Nanobody_id
    try:
        # True positive
        if row.true_label == 1 and row.pred_label == 1:
            obs_attr = get_obs_attr_from_cdrs(row.CDRs, target=1)

            record = {
                "Nanobody_id": seq_id,
                "True_label": int(row.true_label),
                "Pred_label": int(row.pred_label),
                "Pred_prob_1": float(row.pred_prob_1),
                "Sequence": row.Sequence,
                "Aligned_sequence": row["Aligned Sequence"],
                "CDRs": row.CDRs,
                "Round": row.Round,
                "explanation_matrix": obs_attr.tolist()
            }

            positive_explanations.append(record)

        # True negative
        elif row.true_label == 0 and row.pred_label == 0:
            obs_attr = get_obs_attr_from_cdrs(row.CDRs, target=0)

            record = {
                "Nanobody_id": seq_id,
                "True_label": int(row.true_label),
                "Pred_label": int(row.pred_label),
                "Pred_prob_1": float(row.pred_prob_1),
                "Sequence": row.Sequence,
                "Aligned_sequence": row["Aligned Sequence"],
                "CDRs": row.CDRs,
                "Round": row.Round,
                "explanation_matrix": obs_attr.tolist()
            }

            negative_explanations.append(record)

        else:
            continue

    except Exception as e:
        print("Skipping", seq_id, e)

print("Number of positive explanations (TP):", len(positive_explanations))
print("Number of negative explanations (TN):", len(negative_explanations))

if len(positive_explanations) > 0:
    print("Example TP CDR:", positive_explanations[0]["CDRs"][:30])
    print("TP attr shape:", np.array(positive_explanations[0]["explanation_matrix"]).shape)

if len(negative_explanations) > 0:
    print("Example TN CDR:", negative_explanations[0]["CDRs"][:30])
    print("TN attr shape:", np.array(negative_explanations[0]["explanation_matrix"]).shape)

with open("/scratch/5269326/projects/myproj/outputs/Round3/good_positive_explanations_70_test.pkl", "wb") as f:
    pickle.dump(positive_explanations, f)

with open("/scratch/5269326/projects/myproj/outputs/Round3/good_negative_explanations_70_test.pkl", "wb") as f:
    pickle.dump(negative_explanations, f)

print("Everything is done")

Number of positive explanations (TP): 9901
Number of negative explanations (TN): 12415
Example TP CDR: GTIS-------DPVYIARG---GITAAVDG
TP attr shape: (54, 21)
Example TN CDR: GYIS-------NQYGIGYG---GNTAAVIG
TN attr shape: (54, 21)


=========== Everything is done ==========
